<hr style="border: 2px solid #3B6EA5; margin-top: 10px;">
<br>
<h1 style="font-family:verdana; font-size:26px"><center>WikiNav V2, Internal Data + Pretrained Models</center></h1>
<h3 style="font-family:verdana; font-size:14px; color:gray"><center>Candidate reconstruction, transductive audit, visual features, semantic reranking, and fold-safe ensemble</center></h3>
<hr style="border: 2px solid #3B6EA5; margin-top: 10px;">

Notebook ini merupakan pengembangan menyeluruh dari `wikinav_v1`.

Aturan yang dipakai:

- Tidak menggunakan SNAP, Wikispeedia public graph, Wikipedia live, public paths, atau dataset eksternal lain.
- Boleh menggunakan pretrained model.
- Seluruh data task berasal dari `articles.csv`, `categories.csv`, train/test states, dan screenshots kompetisi.
- Candidate recall diukur terpisah dari ranking accuracy.
- Candidate generation validation dibuat fold-safe.

# Daftar Isi

1. Setup dan konfigurasi
2. Load data dan audit struktur
3. Audit state sequence gabungan
4. Text, category, dan pretrained representations
5. Internal screenshot link extraction
6. Internal graph dan graph features
7. Candidate generation
8. Feature engineering
9. Fold-safe validation dan ensemble
10. Final training dan submission
11. Error analysis dan output

## 1. Setup dan konfigurasi

### Dependencies

Install the project dependencies from the repository root before running this notebook:

```bash
pip install -r requirements.txt
```


In [ ]:
from __future__ import annotations

import gc
import html
import json
import math
import os
import pickle
import random
import re
import time
import unicodedata
import warnings
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter

try:
    import torch
except Exception:
    torch = None

import scipy.sparse as sp
from scipy.sparse.csgraph import shortest_path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold, GroupKFold

warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None

try:
    import lightgbm as lgb
except Exception as exc:
    lgb = None
    print("[warn] LightGBM unavailable:", exc)

try:
    import cv2
except Exception:
    cv2 = None

try:
    import easyocr
except Exception as exc:
    easyocr = None
    print("[warn] EasyOCR unavailable:", exc)

try:
    from rapidfuzz import fuzz, process
except Exception:
    fuzz = None
    process = None

try:
    from sentence_transformers import SentenceTransformer, CrossEncoder
except Exception as exc:
    SentenceTransformer = None
    CrossEncoder = None
    print("[warn] sentence-transformers unavailable:", exc)


class CFG:
    SEED = 42
    N_FOLDS = 5

    DATA_DIR = None
    OUTPUT_DIR = None

    FAST_MODE = False

    # OCR
    RUN_OCR = True
    OCR_SCOPE = "current_only"       # current_only or all
    OCR_MAX_PAGES = None
    OCR_STRIP_HEIGHT = 1800
    OCR_STRIP_OVERLAP = 80
    OCR_CANVAS_SIZE = 4096
    OCR_MAG_RATIO = 1.35
    OCR_MIN_CONF = 0.10
    OCR_MATCH_CUTOFF = 76
    OCR_TOPK_MATCHES = 2
    OCR_CHECKPOINT_EVERY = 20

    # Candidate generation
    TRANS_TARGET_TOPK = 4
    MIN_CANDIDATES = 2
    SEQ_OFFSETS = (1, 2, 3, 4, 5)
    SEQ_MIN_SUPPORT = 30
    SEQ_MIN_PRECISION = 0.80

    # Pretrained semantic models
    USE_BI_ENCODER = True
    BI_ENCODER_MODEL = "sentence-transformers/all-mpnet-base-v2"
    USE_CROSS_ENCODER = True
    CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    EMB_BATCH_SIZE = 256
    CROSS_BATCH_SIZE = 256

    # Graph
    DIST_CAP = 99
    COMPUTE_UNDIRECTED_DISTANCE = True

    # LightGBM
    MAX_ROUNDS = 1600
    EARLY_STOPPING = 100
    FINAL_ROUND_MULT = 1.08
    FINAL_SEEDS = (42, 43)

    LTR_PARAMS = {
        "objective": "lambdarank",
        "metric": "ndcg",
        "ndcg_eval_at": [1],
        "learning_rate": 0.035,
        "num_leaves": 63,
        "min_data_in_leaf": 25,
        "feature_fraction": 0.90,
        "bagging_fraction": 0.90,
        "bagging_freq": 1,
        "lambda_l1": 0.05,
        "lambda_l2": 2.0,
        "max_depth": -1,
        "verbosity": -1,
        "seed": 42,
    }

    BIN_PARAMS = {
        "objective": "binary",
        "metric": "binary_logloss",
        "learning_rate": 0.035,
        "num_leaves": 63,
        "min_data_in_leaf": 35,
        "feature_fraction": 0.90,
        "bagging_fraction": 0.90,
        "bagging_freq": 1,
        "lambda_l1": 0.05,
        "lambda_l2": 2.0,
        "verbosity": -1,
        "seed": 42,
    }


random.seed(CFG.SEED)
np.random.seed(CFG.SEED)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 100)


def find_data_dir() -> Path:
    required = {
        "articles.csv",
        "categories.csv",
        "states_train.csv",
        "states_test.csv",
        "sample_submission.csv",
    }

    if CFG.DATA_DIR:
        p = Path(CFG.DATA_DIR)
        if not p.exists():
            raise FileNotFoundError(p)
        return p

    candidates = [
        Path("/kaggle/input/datathon-task-2/dataset-task2"),
        Path("dataset"),
        Path("./dataset"),
        Path("../dataset"),
    ]

    for p in candidates:
        if p.exists() and required.issubset({x.name for x in p.iterdir() if x.is_file()}):
            return p

    root = Path("/kaggle/input")
    if root.exists():
        for hit in sorted(root.rglob("states_train.csv")):
            p = hit.parent
            if required.issubset({x.name for x in p.iterdir() if x.is_file()}):
                return p

    raise FileNotFoundError("Dataset directory not found.")


DATA_DIR = find_data_dir()

SCREEN_DIR = None
for p in [
    DATA_DIR / "screenshots",
    DATA_DIR.parent / "screenshots",
    Path("screenshots"),
]:
    if p.exists():
        SCREEN_DIR = p
        break

if CFG.OUTPUT_DIR:
    OUTPUT_DIR = Path(CFG.OUTPUT_DIR)
else:
    kaggle_working = Path("/kaggle/working")
    OUTPUT_DIR = (
        kaggle_working / "wikinav_v2"
        if kaggle_working.exists()
        else Path("outputs") / "wikinav_v2"
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = OUTPUT_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR   :", DATA_DIR)
print("SCREEN_DIR :", SCREEN_DIR)
print("OUTPUT_DIR :", OUTPUT_DIR)
print("GPU OCR    :", easyocr is not None)
print("LightGBM   :", lgb is not None)

## 2. Load data dan audit struktur

In [ ]:
articles = pd.read_csv(DATA_DIR / "articles.csv")
categories = pd.read_csv(DATA_DIR / "categories.csv")
train = pd.read_csv(DATA_DIR / "states_train.csv")
test = pd.read_csv(DATA_DIR / "states_test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

articles = articles.sort_values("article_id").reset_index(drop=True)
assert np.array_equal(articles["article_id"].to_numpy(), np.arange(len(articles))), "article_id must be contiguous 0..N-1"

required_columns = {
    "articles": {"article_id", "title"},
    "categories": {"article_id", "category"},
    "train": {"state_id", "current_article_id", "target_article_id", "next_article_id"},
    "test": {"state_id", "current_article_id", "target_article_id"},
    "sample_submission": {"state_id", "predicted_next_article_id"},
}

for name, frame in {
    "articles": articles,
    "categories": categories,
    "train": train,
    "test": test,
    "sample_submission": sample_submission,
}.items():
    missing = required_columns[name] - set(frame.columns)
    if missing:
        raise ValueError(f"{name} missing columns: {sorted(missing)}")

article_ids = set(articles["article_id"].astype(int))
for frame_name, frame in [("train", train), ("test", test)]:
    for col in ["current_article_id", "target_article_id"]:
        bad = ~frame[col].isin(article_ids)
        if bad.any():
            raise ValueError(f"{frame_name}.{col} contains unknown article IDs")
if (~train["next_article_id"].isin(article_ids)).any():
    raise ValueError("train.next_article_id contains unknown article IDs")

id_to_title = articles.set_index("article_id")["title"].astype(str).to_dict()
N_ART = len(articles)

inventory = pd.DataFrame([
    {
        "table": name,
        "rows": len(frame),
        "columns": len(frame.columns),
        "missing": int(frame.isna().sum().sum()),
        "duplicates": int(frame.duplicated().sum()),
    }
    for name, frame in {
        "articles": articles,
        "categories": categories,
        "train": train,
        "test": test,
        "sample_submission": sample_submission,
    }.items()
])

display(inventory)
print("Unique train current:", train.current_article_id.nunique())
print("Unique test current :", test.current_article_id.nunique())
print("Current overlap     :", len(set(train.current_article_id) & set(test.current_article_id)))
print("Exact pair overlap  :", len(
    set(zip(train.current_article_id, train.target_article_id))
    & set(zip(test.current_article_id, test.target_article_id))
))

### High-value checks

Sebelum modeling, notebook menguji dua kemungkinan gain besar:

1. Exact `(current, target)` repetition.
2. Urutan state gabungan train-test berdasarkan `state_id`.

In [ ]:
pair_stats = (
    train.groupby(["current_article_id", "target_article_id"])["next_article_id"]
    .agg(
        rows="size",
        unique_next="nunique",
        modal_next=lambda s: int(s.value_counts().index[0]),
        modal_share=lambda s: float(s.value_counts(normalize=True).iloc[0]),
    )
    .reset_index()
)

print("Repeated pair ratio :", (pair_stats.rows > 1).mean())
print("Ambiguous pair ratio:", (pair_stats.unique_next > 1).mean())
display(pair_stats.sort_values(["rows", "modal_share"], ascending=[False, False]).head(20))

pair_stats.to_csv(OUTPUT_DIR / "pair_stats.csv", index=False)

## 3. Audit state sequence gabungan train-test

In [ ]:
train_cov = train[["state_id", "current_article_id", "target_article_id"]].copy()
train_cov["split"] = "train"
train_cov["true_next"] = train["next_article_id"].values

test_cov = test[["state_id", "current_article_id", "target_article_id"]].copy()
test_cov["split"] = "test"
test_cov["true_next"] = np.nan

all_states = pd.concat([train_cov, test_cov], ignore_index=True)
all_states_by_id = all_states.set_index("state_id").sort_index()

sequence_rows = []
sequence_candidate_maps = {}

for offset in CFG.SEQ_OFFSETS:
    future = all_states_by_id[["current_article_id", "target_article_id"]].copy()
    future.index = future.index - offset
    future = future.rename(columns={
        "current_article_id": "future_current",
        "target_article_id": "future_target",
    })

    audit = train_cov.merge(
        future,
        left_on="state_id",
        right_index=True,
        how="left",
    )

    eligible = audit["future_current"].notna() & (
        audit["target_article_id"] == audit["future_target"]
    )
    support = int(eligible.sum())
    precision = float(
        (
            audit.loc[eligible, "future_current"].astype(int)
            == audit.loc[eligible, "true_next"].astype(int)
        ).mean()
    ) if support else 0.0

    sequence_rows.append({
        "offset": offset,
        "support_train": support,
        "precision_train": precision,
        "coverage_train": support / max(1, len(train)),
    })

    full = all_states.reset_index(drop=True).merge(
        future,
        left_on="state_id",
        right_index=True,
        how="left",
    )
    valid = full["future_current"].notna() & (
        full["target_article_id"] == full["future_target"]
    )
    sequence_candidate_maps[offset] = {
        int(sid): int(cand)
        for sid, cand in full.loc[valid, ["state_id", "future_current"]].itertuples(index=False)
    }

sequence_audit = pd.DataFrame(sequence_rows)
display(sequence_audit)

valid_offsets = sequence_audit[
    (sequence_audit.support_train >= CFG.SEQ_MIN_SUPPORT)
    & (sequence_audit.precision_train >= CFG.SEQ_MIN_PRECISION)
].sort_values(["precision_train", "support_train"], ascending=False)

if len(valid_offsets):
    BEST_SEQ_OFFSET = int(valid_offsets.iloc[0].offset)
    SEQ_RULE_PRECISION = float(valid_offsets.iloc[0].precision_train)
    SEQ_MAP = sequence_candidate_maps[BEST_SEQ_OFFSET]
else:
    BEST_SEQ_OFFSET = None
    SEQ_RULE_PRECISION = 0.0
    SEQ_MAP = {}

print("Selected sequence offset:", BEST_SEQ_OFFSET)
print("Sequence precision       :", SEQ_RULE_PRECISION)
print("Sequence test coverage   :", np.mean(test.state_id.isin(SEQ_MAP)) if SEQ_MAP else 0.0)

sequence_audit.to_csv(OUTPUT_DIR / "state_sequence_audit.csv", index=False)

### Transductive target-current signal

Current dan target pada test bukan label. Karena itu kita boleh menggunakan seluruh train dan test untuk menghitung seberapa sering suatu artikel muncul sebagai current untuk target tertentu.

In [ ]:
target_current_counts_all = (
    all_states.groupby(["target_article_id", "current_article_id"])
    .size()
    .to_dict()
)

target_current_top = {}
for target, sub in (
    all_states.groupby(["target_article_id", "current_article_id"])
    .size()
    .rename("count")
    .reset_index()
    .groupby("target_article_id")
):
    target_current_top[int(target)] = (
        sub.sort_values("count", ascending=False)
        .head(CFG.TRANS_TARGET_TOPK)["current_article_id"]
        .astype(int)
        .tolist()
    )

print("Targets with transductive current pools:", len(target_current_top))

## 4. Text, category, dan pretrained representations

In [ ]:
def normalize_text(value: Any) -> str:
    value = html.unescape("" if value is None else str(value))
    value = unicodedata.normalize("NFKD", value)
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    value = value.lower()
    value = value.replace("&", " and ")
    value = re.sub(r"[_/]+", " ", value)
    value = re.sub(r"[^a-z0-9\s-]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def title_aliases(title: str) -> set[str]:
    aliases = {normalize_text(title)}
    aliases.add(normalize_text(re.sub(r"\s*\([^)]*\)\s*$", "", title)))
    aliases.add(normalize_text(title.replace("-", " ")))
    if title.lower().startswith("the "):
        aliases.add(normalize_text(title[4:]))
    return {x for x in aliases if len(x) >= 2}


category_map = (
    categories.groupby("article_id")["category"]
    .agg(lambda s: sorted(set(str(x) for x in s.dropna())))
    .to_dict()
)

article_text = []
for aid in articles.article_id:
    cats = category_map.get(int(aid), [])
    article_text.append(
        f"Title: {id_to_title[int(aid)]}. Categories: {'; '.join(cats)}"
    )

article_text = np.asarray(article_text, dtype=object)

alias_to_ids = defaultdict(list)
for aid, title in id_to_title.items():
    for alias in title_aliases(title):
        alias_to_ids[alias].append(int(aid))
alias_choices = list(alias_to_ids.keys())

def category_parts(aid: int) -> set[str]:
    return set(category_map.get(int(aid), []))

def category_top(aid: int) -> set[str]:
    out = set()
    for cat in category_map.get(int(aid), []):
        parts = cat.split(".")
        if len(parts) >= 2:
            out.add(parts[1])
    return out

def common_category_prefix_depth(aid_a: int, aid_b: int) -> int:
    best = 0
    for a in category_map.get(int(aid_a), []):
        pa = a.split(".")
        for b in category_map.get(int(aid_b), []):
            pb = b.split(".")
            depth = 0
            for xa, xb in zip(pa, pb):
                if xa != xb:
                    break
                depth += 1
            best = max(best, depth)
    return best

word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    sublinear_tf=True,
    norm="l2",
)
X_word = word_vectorizer.fit_transform(article_text)

char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    sublinear_tf=True,
    norm="l2",
)
X_char = char_vectorizer.fit_transform(
    [normalize_text(id_to_title[int(aid)]) for aid in articles.article_id]
)

EMB = None
emb_cache = CACHE_DIR / "article_embeddings.npy"

if CFG.USE_BI_ENCODER and SentenceTransformer is not None:
    if emb_cache.exists():
        EMB = np.load(emb_cache)
        print("Loaded embedding cache:", EMB.shape)
    else:
        model = SentenceTransformer(CFG.BI_ENCODER_MODEL)
        EMB = np.asarray(
            model.encode(
                article_text.tolist(),
                batch_size=CFG.EMB_BATCH_SIZE,
                normalize_embeddings=True,
                show_progress_bar=True,
            ),
            dtype=np.float32,
        )
        np.save(emb_cache, EMB)
        del model
        gc.collect()
        print("Saved embedding cache:", EMB.shape)
else:
    print("Bi-encoder disabled or unavailable.")

## 5. Internal screenshot link extraction

In [ ]:
def blue_mask_rgb(arr: np.ndarray) -> np.ndarray:
    r = arr[..., 0].astype(np.int16)
    g = arr[..., 1].astype(np.int16)
    b = arr[..., 2].astype(np.int16)

    mask_a = (b > 105) & (b - r > 35) & (b - g > 12)
    mask_b = (b > 125) & (r < 135) & (g < 170)
    return mask_a | mask_b


def blue_only_image(arr: np.ndarray) -> np.ndarray:
    mask = blue_mask_rgb(arr)

    if cv2 is not None:
        expanded = cv2.dilate(mask.astype(np.uint8), np.ones((2, 2), np.uint8), iterations=1).astype(bool)
    else:
        expanded = np.asarray(
            Image.fromarray((mask.astype(np.uint8) * 255)).filter(ImageFilter.MaxFilter(3))
        ) > 0

    out = np.full_like(arr, 255)
    out[expanded] = arr[expanded]
    return out


def match_anchor_to_titles(text: str) -> list[tuple[int, float]]:
    norm = normalize_text(text)
    if len(norm) < 2:
        return []

    if norm in alias_to_ids:
        return [(aid, 100.0) for aid in alias_to_ids[norm][:CFG.OCR_TOPK_MATCHES]]

    if process is None or fuzz is None:
        return []

    hits = process.extract(
        norm,
        alias_choices,
        scorer=fuzz.WRatio,
        limit=max(5, CFG.OCR_TOPK_MATCHES * 2),
        score_cutoff=CFG.OCR_MATCH_CUTOFF,
    )

    candidates = {}
    for alias, score, _ in hits:
        for aid in alias_to_ids[alias]:
            candidates[int(aid)] = max(float(score), candidates.get(int(aid), 0.0))

    return sorted(candidates.items(), key=lambda x: x[1], reverse=True)[:CFG.OCR_TOPK_MATCHES]


OCR_CACHE_PATH = CACHE_DIR / "ocr_link_records.json"

if OCR_CACHE_PATH.exists():
    raw_cache = json.loads(OCR_CACHE_PATH.read_text(encoding="utf-8"))
    OCR_RECORDS = {int(k): v for k, v in raw_cache.items()}
else:
    OCR_RECORDS = {}

ocr_reader = None
if CFG.RUN_OCR and easyocr is not None and SCREEN_DIR is not None:
    try:
        use_gpu = bool(torch is not None and torch.cuda.is_available())
        ocr_reader = easyocr.Reader(["en"], gpu=use_gpu, verbose=False)
    except Exception as exc:
        print("[warn] EasyOCR initialization failed:", exc)


def extract_page_ocr_records(aid: int) -> list[dict[str, Any]]:
    path = SCREEN_DIR / f"{aid}.png"
    if not path.exists() or ocr_reader is None:
        return []

    arr = np.asarray(Image.open(path).convert("RGB"))
    height, width = arr.shape[:2]
    step = max(200, CFG.OCR_STRIP_HEIGHT - CFG.OCR_STRIP_OVERLAP)

    raw_hits = []

    for y0 in range(0, height, step):
        y1 = min(height, y0 + CFG.OCR_STRIP_HEIGHT)
        strip = arr[y0:y1]

        if blue_mask_rgb(strip).sum() < 10:
            continue

        masked = blue_only_image(strip)

        try:
            results = ocr_reader.readtext(
                masked,
                detail=1,
                paragraph=False,
                decoder="greedy",
                canvas_size=CFG.OCR_CANVAS_SIZE,
                mag_ratio=CFG.OCR_MAG_RATIO,
            )
        except Exception:
            results = []

        for bbox, text, conf in results:
            conf = float(conf)
            if conf < CFG.OCR_MIN_CONF:
                continue

            ys = [float(p[1]) for p in bbox]
            xs = [float(p[0]) for p in bbox]
            global_y = y0 + min(ys)
            y_norm = global_y / max(1, height)
            x_norm = min(xs) / max(1, width)

            matches = match_anchor_to_titles(text)
            for candidate_id, match_score in matches:
                raw_hits.append({
                    "candidate_id": int(candidate_id),
                    "anchor_text": str(text),
                    "ocr_conf": conf,
                    "match_score": float(match_score),
                    "y_norm": float(y_norm),
                    "x_norm": float(x_norm),
                })

    dedup = {}
    for rec in raw_hits:
        candidate = rec["candidate_id"]
        if candidate == aid:
            continue
        prev = dedup.get(candidate)
        quality = rec["ocr_conf"] * rec["match_score"]
        if prev is None or quality > prev["ocr_conf"] * prev["match_score"]:
            dedup[candidate] = dict(rec)
            dedup[candidate]["occurrences"] = 1
        else:
            prev["occurrences"] += 1
            prev["y_norm"] = min(prev["y_norm"], rec["y_norm"])
            prev["x_norm"] = min(prev["x_norm"], rec["x_norm"])

    ordered = sorted(dedup.values(), key=lambda x: (x["y_norm"], x["x_norm"]))
    denom = max(1, len(ordered) - 1)
    for idx, rec in enumerate(ordered):
        rec["order_norm"] = idx / denom
        rec["first_screen"] = float(rec["y_norm"] <= 0.18)

    return ordered

In [ ]:
if SCREEN_DIR is None:
    print("Screenshot directory not found, OCR stage skipped.")
elif ocr_reader is None:
    print("OCR reader unavailable, existing cache only will be used.")
else:
    if CFG.OCR_SCOPE == "all":
        page_ids = sorted(article_ids)
    else:
        page_ids = sorted(set(train.current_article_id) | set(test.current_article_id))

    if CFG.OCR_MAX_PAGES is not None:
        page_ids = page_ids[:CFG.OCR_MAX_PAGES]

    if CFG.FAST_MODE:
        page_ids = page_ids[: min(200, len(page_ids))]

    todo = [aid for aid in page_ids if int(aid) not in OCR_RECORDS]
    print("OCR pages total :", len(page_ids))
    print("OCR pages cached:", len(page_ids) - len(todo))
    print("OCR pages todo  :", len(todo))

    started = time.time()
    for idx, aid in enumerate(todo, start=1):
        OCR_RECORDS[int(aid)] = extract_page_ocr_records(int(aid))

        if idx % CFG.OCR_CHECKPOINT_EVERY == 0:
            OCR_CACHE_PATH.write_text(
                json.dumps(OCR_RECORDS, ensure_ascii=False),
                encoding="utf-8",
            )
            print(
                f"{idx}/{len(todo)} pages, "
                f"{(time.time() - started) / 60:.1f} minutes"
            )

    OCR_CACHE_PATH.write_text(
        json.dumps(OCR_RECORDS, ensure_ascii=False),
        encoding="utf-8",
    )

print("OCR pages available:", len(OCR_RECORDS))

In [ ]:
OCR_META = defaultdict(dict)
OCR_ADJ = defaultdict(set)

for aid, records in OCR_RECORDS.items():
    for rec in records:
        candidate = int(rec["candidate_id"])
        OCR_ADJ[int(aid)].add(candidate)
        OCR_META[int(aid)][candidate] = rec

ocr_sizes = np.array([len(OCR_ADJ.get(int(aid), set())) for aid in set(train.current_article_id)])
ocr_train_recall = np.mean([
    int(next_id) in OCR_ADJ.get(int(current), set())
    for current, next_id in train[["current_article_id", "next_article_id"]].itertuples(index=False)
])

print("OCR edge count           :", sum(len(v) for v in OCR_ADJ.values()))
print("OCR mean candidates/page :", float(ocr_sizes.mean()) if len(ocr_sizes) else 0.0)
print("OCR train-label recall   :", ocr_train_recall)

ocr_summary = pd.DataFrame([{
    "pages": len(OCR_RECORDS),
    "edges": sum(len(v) for v in OCR_ADJ.values()),
    "mean_candidates_train_current": float(ocr_sizes.mean()) if len(ocr_sizes) else 0.0,
    "train_label_recall": float(ocr_train_recall),
}])
display(ocr_summary)
ocr_summary.to_csv(OUTPUT_DIR / "ocr_summary.csv", index=False)

### Catatan

Train-label recall OCR tidak boleh dinilai setelah union dengan empirical train edges karena union tersebut otomatis memasukkan label train. Angka OCR-only merupakan ukuran yang jujur untuk kualitas screenshot reconstruction.

## 6. Internal graph dan graph features

In [ ]:
rows = []
cols = []

for source, neighbors in OCR_ADJ.items():
    for target_id in neighbors:
        rows.append(int(source))
        cols.append(int(target_id))

if rows:
    A_OCR = sp.csr_matrix(
        (
            np.ones(len(rows), dtype=np.int8),
            (np.asarray(rows), np.asarray(cols)),
        ),
        shape=(N_ART, N_ART),
    )
else:
    A_OCR = sp.csr_matrix((N_ART, N_ART), dtype=np.int8)

OUT_DEG = np.asarray(A_OCR.sum(axis=1)).ravel().astype(np.float32)
IN_DEG = np.asarray(A_OCR.sum(axis=0)).ravel().astype(np.float32)

ALL_TARGETS = sorted(set(train.target_article_id) | set(test.target_article_id))
TARGET_TO_ROW = {int(t): i for i, t in enumerate(ALL_TARGETS)}

if A_OCR.nnz:
    started = time.time()
    D_DIR = shortest_path(
        A_OCR.T,
        directed=True,
        unweighted=True,
        indices=ALL_TARGETS,
    )
    D_DIR = np.where(np.isfinite(D_DIR), D_DIR, CFG.DIST_CAP).astype(np.int16)

    if CFG.COMPUTE_UNDIRECTED_DISTANCE:
        A_UNDIR = A_OCR.maximum(A_OCR.T)
        D_UNDIR = shortest_path(
            A_UNDIR,
            directed=False,
            unweighted=True,
            indices=ALL_TARGETS,
        )
        D_UNDIR = np.where(np.isfinite(D_UNDIR), D_UNDIR, CFG.DIST_CAP).astype(np.int16)
    else:
        D_UNDIR = np.full_like(D_DIR, CFG.DIST_CAP)

    print("Distance matrices:", D_DIR.shape, f"in {time.time() - started:.1f}s")
else:
    D_DIR = np.full((len(ALL_TARGETS), N_ART), CFG.DIST_CAP, dtype=np.int16)
    D_UNDIR = np.full_like(D_DIR, CFG.DIST_CAP)
    print("OCR graph empty, distance features capped.")

print("OCR graph nodes with links:", int((OUT_DEG > 0).sum()))
print("OCR graph edges           :", int(A_OCR.nnz))

## 7. Candidate generation

In [ ]:
def empirical_adjacency(states: pd.DataFrame) -> dict[int, set[int]]:
    adj = defaultdict(set)
    for current, next_id in states[
        ["current_article_id", "next_article_id"]
    ].itertuples(index=False):
        adj[int(current)].add(int(next_id))
    return adj


def candidate_rows_for_states(
    states: pd.DataFrame,
    empirical_adj: dict[int, set[int]],
    with_label: bool,
) -> pd.DataFrame:
    rows = []

    for row in states.itertuples(index=False):
        state_id = int(row.state_id)
        current = int(row.current_article_id)
        target = int(row.target_article_id)
        actual = int(row.next_article_id) if with_label else None

        source_map = defaultdict(lambda: {
            "source_ocr": 0.0,
            "source_empirical": 0.0,
            "source_sequence": 0.0,
            "source_transductive": 0.0,
        })

        for candidate in OCR_ADJ.get(current, set()):
            source_map[int(candidate)]["source_ocr"] = 1.0

        for candidate in empirical_adj.get(current, set()):
            source_map[int(candidate)]["source_empirical"] = 1.0

        seq_candidate = SEQ_MAP.get(state_id)
        if seq_candidate is not None and seq_candidate != current:
            source_map[int(seq_candidate)]["source_sequence"] = 1.0

        if len(source_map) < CFG.MIN_CANDIDATES:
            for candidate in target_current_top.get(target, []):
                if int(candidate) != current:
                    source_map[int(candidate)]["source_transductive"] = 1.0

        if not source_map:
            source_map[target]["source_transductive"] = 1.0

        for candidate, flags in source_map.items():
            rec = OCR_META.get(current, {}).get(candidate, {})
            rows.append({
                "state_id": state_id,
                "cur": current,
                "tgt": target,
                "cand": int(candidate),
                "label": float(candidate == actual) if with_label else np.nan,
                **flags,
                "ocr_conf": float(rec.get("ocr_conf", 0.0)),
                "ocr_match_score": float(rec.get("match_score", 0.0)),
                "ocr_y_norm": float(rec.get("y_norm", 1.0)),
                "ocr_x_norm": float(rec.get("x_norm", 1.0)),
                "ocr_order_norm": float(rec.get("order_norm", 1.0)),
                "ocr_occurrences": float(rec.get("occurrences", 0.0)),
                "ocr_first_screen": float(rec.get("first_screen", 0.0)),
            })

    return pd.DataFrame(rows)


def candidate_recall(pairs: pd.DataFrame) -> float:
    if "label" not in pairs:
        return np.nan
    label_per_state = pairs.groupby("state_id")["label"].max()
    return float(label_per_state.mean())


full_empirical_adj = empirical_adjacency(train)
pairs_full_train_preview = candidate_rows_for_states(
    train,
    full_empirical_adj,
    with_label=True,
)

print("Full-train candidate recall is expected to be 1.0 because empirical labels are included.")
print("Full-train candidate recall:", candidate_recall(pairs_full_train_preview))
print("Mean candidate count:", pairs_full_train_preview.groupby("state_id").size().mean())
del pairs_full_train_preview
gc.collect()

### Fold-safe rule

Pada CV, validation candidate set dibangun hanya dari:

- OCR graph;
- empirical edges dari training fold;
- state-sequence candidate;
- target-current transductive fallback.

Validation label tidak pernah dipakai untuk membentuk candidate set.

### Grouped-current candidate recall diagnostic

Protocol ini menahan seluruh halaman `current` dari training fold. Karena itu empirical edges tidak dapat membantu validation current. Angka ini mengukur ketergantungan pipeline pada OCR dan signal transduktif untuk halaman yang benar-benar belum terlihat.

In [ ]:
group_candidate_rows = []

n_group_splits = min(5, train["current_article_id"].nunique())
group_cv = GroupKFold(n_splits=n_group_splits)

for fold, (train_idx, valid_idx) in enumerate(
    group_cv.split(train, groups=train["current_article_id"])
):
    fold_train = train.iloc[train_idx]
    fold_valid = train.iloc[valid_idx]

    fold_empirical = empirical_adjacency(fold_train)
    fold_pairs = candidate_rows_for_states(
        fold_valid,
        fold_empirical,
        with_label=True,
    )

    group_candidate_rows.append({
        "fold": fold,
        "candidate_recall": candidate_recall(fold_pairs),
        "mean_candidates": fold_pairs.groupby("state_id").size().mean(),
        "validation_states": len(fold_valid),
        "validation_unique_current": fold_valid.current_article_id.nunique(),
    })

group_candidate_diagnostic = pd.DataFrame(group_candidate_rows)
display(group_candidate_diagnostic)

group_candidate_diagnostic.to_csv(
    OUTPUT_DIR / "group_current_candidate_recall.csv",
    index=False,
)

## 8. Feature engineering

In [ ]:
def sparse_pair_similarity(matrix, a_ids, b_ids) -> np.ndarray:
    return np.asarray(
        matrix[np.asarray(a_ids)].multiply(matrix[np.asarray(b_ids)]).sum(axis=1)
    ).ravel().astype(np.float32)


def embedding_pair_similarity(a_ids, b_ids) -> np.ndarray:
    if EMB is None:
        return np.zeros(len(a_ids), dtype=np.float32)
    a = EMB[np.asarray(a_ids)]
    b = EMB[np.asarray(b_ids)]
    return np.sum(a * b, axis=1).astype(np.float32)


def set_jaccard(a: set[str], b: set[str]) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def add_static_features(pairs: pd.DataFrame) -> pd.DataFrame:
    pairs = pairs.copy()

    cur = pairs["cur"].astype(int).to_numpy()
    tgt = pairs["tgt"].astype(int).to_numpy()
    cand = pairs["cand"].astype(int).to_numpy()
    trow = np.array([TARGET_TO_ROW[int(x)] for x in tgt], dtype=np.int32)

    pairs["is_target"] = (cand == tgt).astype(np.float32)
    pairs["seq_match"] = np.array([
        float(SEQ_MAP.get(int(sid), -1) == int(c))
        for sid, c in pairs[["state_id", "cand"]].itertuples(index=False)
    ], dtype=np.float32)

    pairs["trans_target_current_count"] = np.array([
        target_current_counts_all.get((int(t), int(c)), 0)
        for t, c in zip(tgt, cand)
    ], dtype=np.float32)

    pairs["out_deg"] = OUT_DEG[cand]
    pairs["in_deg"] = IN_DEG[cand]

    pairs["cand_dist_dir"] = D_DIR[trow, cand].astype(np.float32)
    pairs["cur_dist_dir"] = D_DIR[trow, cur].astype(np.float32)
    pairs["progress_dir"] = pairs["cur_dist_dir"] - pairs["cand_dist_dir"]
    pairs["cand_reachable_dir"] = (pairs["cand_dist_dir"] < CFG.DIST_CAP).astype(np.float32)

    pairs["cand_dist_undir"] = D_UNDIR[trow, cand].astype(np.float32)
    pairs["cur_dist_undir"] = D_UNDIR[trow, cur].astype(np.float32)
    pairs["progress_undir"] = pairs["cur_dist_undir"] - pairs["cand_dist_undir"]

    pairs["emb_sim_ct"] = embedding_pair_similarity(cand, tgt)
    pairs["emb_sim_cs"] = embedding_pair_similarity(cand, cur)
    pairs["emb_sim_st"] = embedding_pair_similarity(cur, tgt)

    pairs["word_tfidf_ct"] = sparse_pair_similarity(X_word, cand, tgt)
    pairs["char_tfidf_ct"] = sparse_pair_similarity(X_char, cand, tgt)

    cat_j = []
    cat_top_match = []
    cat_prefix = []

    for c, t in zip(cand, tgt):
        cat_j.append(set_jaccard(category_parts(int(c)), category_parts(int(t))))
        cat_top_match.append(float(bool(category_top(int(c)) & category_top(int(t)))))
        cat_prefix.append(float(common_category_prefix_depth(int(c), int(t))))

    pairs["cat_jaccard_ct"] = np.asarray(cat_j, dtype=np.float32)
    pairs["cat_top_match_ct"] = np.asarray(cat_top_match, dtype=np.float32)
    pairs["cat_prefix_depth_ct"] = np.asarray(cat_prefix, dtype=np.float32)

    group = pairs.groupby("state_id", sort=False)

    pairs["n_cand"] = group["cand"].transform("size").astype(np.float32)
    pairs["rank_dist_dir"] = group["cand_dist_dir"].rank(
        method="dense", ascending=True
    ).astype(np.float32)
    pairs["rank_emb_ct"] = group["emb_sim_ct"].rank(
        method="dense", ascending=False
    ).astype(np.float32)

    min_dist = group["cand_dist_dir"].transform("min")
    pairs["is_argmin_dir"] = (pairs["cand_dist_dir"] == min_dist).astype(np.float32)
    pairs["argmin_size_dir"] = group["is_argmin_dir"].transform("sum").astype(np.float32)

    return pairs


def fit_behavior_counts(states: pd.DataFrame) -> dict[str, dict]:
    states = states.copy()

    article_top = {
        int(aid): sorted(category_top(int(aid)))[0]
        if category_top(int(aid))
        else "__missing__"
        for aid in article_ids
    }

    tmp = states.assign(
        cur_top=states.current_article_id.map(article_top),
        tgt_top=states.target_article_id.map(article_top),
        nxt_top=states.next_article_id.map(article_top),
    )

    return {
        "sc": states.groupby(["current_article_id", "next_article_id"]).size().to_dict(),
        "s": states.groupby("current_article_id").size().to_dict(),
        "tc": states.groupby(["target_article_id", "next_article_id"]).size().to_dict(),
        "t": states.groupby("target_article_id").size().to_dict(),
        "stc": states.groupby(
            ["current_article_id", "target_article_id", "next_article_id"]
        ).size().to_dict(),
        "st": states.groupby(
            ["current_article_id", "target_article_id"]
        ).size().to_dict(),
        "c": states.groupby("next_article_id").size().to_dict(),
        "cat_transition": tmp.groupby(
            ["cur_top", "tgt_top", "nxt_top"]
        ).size().to_dict(),
        "pair_modal": (
            states.groupby(["current_article_id", "target_article_id"])["next_article_id"]
            .agg(
                modal_next=lambda s: int(s.value_counts().index[0]),
                modal_count=lambda s: int(s.value_counts().iloc[0]),
                support="size",
            )
            .to_dict("index")
        ),
    }


def add_behavior_features(pairs: pd.DataFrame, counts: dict[str, dict]) -> pd.DataFrame:
    pairs = pairs.copy()

    sc = counts["sc"]
    s_count = counts["s"]
    tc = counts["tc"]
    t_count = counts["t"]
    stc = counts["stc"]
    st_count = counts["st"]
    c_count = counts["c"]
    cat_transition = counts["cat_transition"]

    article_top = {
        int(aid): sorted(category_top(int(aid)))[0]
        if category_top(int(aid))
        else "__missing__"
        for aid in article_ids
    }

    rows = list(pairs[["cur", "tgt", "cand"]].itertuples(index=False, name=None))

    clicks_sc = []
    rate_sc = []
    clicks_tc = []
    rate_tc = []
    clicks_stc = []
    rate_stc = []
    clicks_c = []
    cat_trans = []

    for cur, tgt, cand in rows:
        cur = int(cur)
        tgt = int(tgt)
        cand = int(cand)

        v_sc = sc.get((cur, cand), 0)
        v_tc = tc.get((tgt, cand), 0)
        v_stc = stc.get((cur, tgt, cand), 0)

        clicks_sc.append(v_sc)
        rate_sc.append((v_sc + 0.5) / (s_count.get(cur, 0) + 2.0))
        clicks_tc.append(v_tc)
        rate_tc.append((v_tc + 0.5) / (t_count.get(tgt, 0) + 2.0))
        clicks_stc.append(v_stc)
        rate_stc.append((v_stc + 0.5) / (st_count.get((cur, tgt), 0) + 2.0))
        clicks_c.append(c_count.get(cand, 0))
        cat_trans.append(
            cat_transition.get(
                (article_top[cur], article_top[tgt], article_top[cand]),
                0,
            )
        )

    pairs["log_clicks_sc"] = np.log1p(clicks_sc).astype(np.float32)
    pairs["rate_sc"] = np.asarray(rate_sc, dtype=np.float32)
    pairs["log_clicks_tc"] = np.log1p(clicks_tc).astype(np.float32)
    pairs["rate_tc"] = np.asarray(rate_tc, dtype=np.float32)
    pairs["log_clicks_stc"] = np.log1p(clicks_stc).astype(np.float32)
    pairs["rate_stc"] = np.asarray(rate_stc, dtype=np.float32)
    pairs["log_clicks_c"] = np.log1p(clicks_c).astype(np.float32)
    pairs["log_cat_transition"] = np.log1p(cat_trans).astype(np.float32)

    return pairs


FEATURES = [
    "source_ocr",
    "source_empirical",
    "source_sequence",
    "source_transductive",
    "ocr_conf",
    "ocr_match_score",
    "ocr_y_norm",
    "ocr_x_norm",
    "ocr_order_norm",
    "ocr_occurrences",
    "ocr_first_screen",
    "is_target",
    "seq_match",
    "trans_target_current_count",
    "out_deg",
    "in_deg",
    "cand_dist_dir",
    "cur_dist_dir",
    "progress_dir",
    "cand_reachable_dir",
    "cand_dist_undir",
    "cur_dist_undir",
    "progress_undir",
    "emb_sim_ct",
    "emb_sim_cs",
    "emb_sim_st",
    "word_tfidf_ct",
    "char_tfidf_ct",
    "cat_jaccard_ct",
    "cat_top_match_ct",
    "cat_prefix_depth_ct",
    "n_cand",
    "rank_dist_dir",
    "rank_emb_ct",
    "is_argmin_dir",
    "argmin_size_dir",
    "log_clicks_sc",
    "rate_sc",
    "log_clicks_tc",
    "rate_tc",
    "log_clicks_stc",
    "rate_stc",
    "log_clicks_c",
    "log_cat_transition",
    "cross_score",
]

### Cross-encoder cache

In [ ]:
class CrossEncoderCache:
    def __init__(self):
        self.path = CACHE_DIR / "cross_encoder_cache.pkl"
        if self.path.exists():
            with open(self.path, "rb") as f:
                self.cache = pickle.load(f)
        else:
            self.cache = {}

        self.model = None

    def ensure_model(self):
        if not CFG.USE_CROSS_ENCODER or CrossEncoder is None:
            return
        if self.model is None:
            self.model = CrossEncoder(
                CFG.CROSS_ENCODER_MODEL,
                max_length=128,
            )

    def score(self, pairs: pd.DataFrame) -> np.ndarray:
        keys = list(
            zip(
                pairs["cur"].astype(int),
                pairs["tgt"].astype(int),
                pairs["cand"].astype(int),
            )
        )

        missing = list(dict.fromkeys(k for k in keys if k not in self.cache))

        if missing and CFG.USE_CROSS_ENCODER and CrossEncoder is not None:
            self.ensure_model()

            text_pairs = []
            for cur, tgt, cand in missing:
                query = (
                    f"Current Wikipedia article: {article_text[cur]} "
                    f"Goal article: {article_text[tgt]}"
                )
                passage = f"Possible next click: {article_text[cand]}"
                text_pairs.append((query, passage))

            scores = self.model.predict(
                text_pairs,
                batch_size=CFG.CROSS_BATCH_SIZE,
                show_progress_bar=True,
            )

            for key, score in zip(missing, scores):
                self.cache[key] = float(score)

            with open(self.path, "wb") as f:
                pickle.dump(self.cache, f)

        return np.asarray(
            [self.cache.get(k, 0.0) for k in keys],
            dtype=np.float32,
        )


cross_cache = CrossEncoderCache()


def complete_features(
    pairs: pd.DataFrame,
    counts: dict[str, dict],
) -> pd.DataFrame:
    pairs = add_static_features(pairs)
    pairs = add_behavior_features(pairs, counts)
    pairs["cross_score"] = cross_cache.score(pairs)
    return pairs

## 9. Fold-safe validation dan ensemble

In [ ]:
def sort_pairs(pairs: pd.DataFrame) -> pd.DataFrame:
    return pairs.sort_values(["state_id", "cand"]).reset_index(drop=True)


def group_sizes(pairs: pd.DataFrame) -> np.ndarray:
    return pairs.groupby("state_id", sort=False).size().to_numpy()


def inverse_group_weights(pairs: pd.DataFrame) -> np.ndarray:
    sizes = pairs.groupby("state_id", sort=False).size()
    mapping = (1.0 / sizes).to_dict()
    return pairs["state_id"].map(mapping).to_numpy(dtype=np.float32)


def train_ltr(
    train_pairs: pd.DataFrame,
    valid_pairs: pd.DataFrame,
    seed: int,
):
    if lgb is None:
        raise RuntimeError("LightGBM is required for V2.")

    params = dict(CFG.LTR_PARAMS)
    params["seed"] = seed

    train_pairs = sort_pairs(train_pairs)
    valid_pairs = sort_pairs(valid_pairs)

    dtrain = lgb.Dataset(
        train_pairs[FEATURES],
        label=train_pairs["label"],
        group=group_sizes(train_pairs),
        feature_name=FEATURES,
    )

    valid_covered_ids = (
        valid_pairs.groupby("state_id")["label"].max()
        .loc[lambda s: s > 0]
        .index
    )
    valid_for_metric = valid_pairs[
        valid_pairs.state_id.isin(valid_covered_ids)
    ].copy()

    if len(valid_for_metric):
        dvalid = lgb.Dataset(
            valid_for_metric[FEATURES],
            label=valid_for_metric["label"],
            group=group_sizes(valid_for_metric),
            reference=dtrain,
            feature_name=FEATURES,
        )

        model = lgb.train(
            params,
            dtrain,
            num_boost_round=CFG.MAX_ROUNDS,
            valid_sets=[dvalid],
            callbacks=[
                lgb.early_stopping(CFG.EARLY_STOPPING, verbose=False),
                lgb.log_evaluation(0),
            ],
        )
        best_iteration = int(model.best_iteration)
    else:
        model = lgb.train(
            params,
            dtrain,
            num_boost_round=400,
        )
        best_iteration = 400

    return model, best_iteration


def train_binary(
    train_pairs: pd.DataFrame,
    valid_pairs: pd.DataFrame,
    seed: int,
):
    params = dict(CFG.BIN_PARAMS)
    params["seed"] = seed

    dtrain = lgb.Dataset(
        train_pairs[FEATURES],
        label=train_pairs["label"],
        weight=inverse_group_weights(train_pairs),
        feature_name=FEATURES,
    )

    dvalid = lgb.Dataset(
        valid_pairs[FEATURES],
        label=valid_pairs["label"],
        weight=inverse_group_weights(valid_pairs),
        reference=dtrain,
        feature_name=FEATURES,
    )

    model = lgb.train(
        params,
        dtrain,
        num_boost_round=CFG.MAX_ROUNDS,
        valid_sets=[dvalid],
        callbacks=[
            lgb.early_stopping(CFG.EARLY_STOPPING, verbose=False),
            lgb.log_evaluation(0),
        ],
    )

    return model, int(model.best_iteration)


def within_state_rank(
    pairs: pd.DataFrame,
    column: str,
    ascending: bool = False,
) -> np.ndarray:
    return (
        pairs.groupby("state_id")[column]
        .rank(method="average", pct=True, ascending=ascending)
        .to_numpy(dtype=np.float32)
    )


def top1_predictions(pairs: pd.DataFrame, score_col: str) -> pd.DataFrame:
    idx = pairs.groupby("state_id")[score_col].idxmax()
    return pairs.loc[idx, ["state_id", "cand", score_col]].copy()


def accuracy_from_predictions(
    predictions: pd.DataFrame,
    truth: dict[int, int],
) -> float:
    actual = predictions["state_id"].map(truth)
    return float((predictions["cand"].astype(int) == actual.astype(int)).mean())


truth_map = dict(zip(train.state_id.astype(int), train.next_article_id.astype(int)))

kf = KFold(
    n_splits=CFG.N_FOLDS,
    shuffle=True,
    random_state=CFG.SEED,
)

fold_assignment = np.full(len(train), -1, dtype=np.int16)
for fold, (_, valid_idx) in enumerate(kf.split(train)):
    fold_assignment[valid_idx] = fold

train_with_fold = train.copy()
train_with_fold["fold"] = fold_assignment

oof_parts = []
fold_metrics = []
best_ltr_iterations = []
best_bin_iterations = []
feature_importance = []

started_cv = time.time()

for fold in range(CFG.N_FOLDS):
    fold_train = train_with_fold[train_with_fold.fold != fold].drop(columns="fold")
    fold_valid = train_with_fold[train_with_fold.fold == fold].drop(columns="fold")

    fold_empirical = empirical_adjacency(fold_train)

    ptr = candidate_rows_for_states(
        fold_train,
        fold_empirical,
        with_label=True,
    )
    pva = candidate_rows_for_states(
        fold_valid,
        fold_empirical,
        with_label=True,
    )

    recall = candidate_recall(pva)
    counts = fit_behavior_counts(fold_train)

    ptr = complete_features(ptr, counts)
    pva = complete_features(pva, counts)

    ltr_model, best_ltr = train_ltr(ptr, pva, CFG.SEED + fold)
    bin_model, best_bin = train_binary(ptr, pva, CFG.SEED + 100 + fold)

    pva["score_ltr"] = ltr_model.predict(
        pva[FEATURES],
        num_iteration=best_ltr,
    )
    pva["score_bin"] = bin_model.predict(
        pva[FEATURES],
        num_iteration=best_bin,
    )

    pva["score_graph"] = (
        -0.70 * pva["rank_dist_dir"]
        + 0.20 * pva["progress_dir"]
        + 0.10 * pva["rate_tc"]
        + 0.12 * pva["is_target"]
        + 0.15 * pva["seq_match"]
    )

    pva["fold"] = fold
    oof_parts.append(pva)

    pred_ltr = top1_predictions(pva, "score_ltr")
    pred_bin = top1_predictions(pva, "score_bin")

    fold_metrics.append({
        "fold": fold,
        "candidate_recall": recall,
        "ltr_accuracy": accuracy_from_predictions(pred_ltr, truth_map),
        "binary_accuracy": accuracy_from_predictions(pred_bin, truth_map),
        "valid_states": len(fold_valid),
        "mean_candidates": pva.groupby("state_id").size().mean(),
    })

    best_ltr_iterations.append(best_ltr)
    best_bin_iterations.append(best_bin)

    feature_importance.append(
        pd.DataFrame({
            "feature": FEATURES,
            "gain": ltr_model.feature_importance(importance_type="gain"),
            "fold": fold,
        })
    )

    print(
        f"fold={fold} recall={recall:.4f} "
        f"ltr={fold_metrics[-1]['ltr_accuracy']:.4f} "
        f"bin={fold_metrics[-1]['binary_accuracy']:.4f} "
        f"time={(time.time() - started_cv) / 60:.1f}m"
    )

    del ptr, pva, ltr_model, bin_model
    gc.collect()

oof = pd.concat(oof_parts, ignore_index=True)
fold_metrics_df = pd.DataFrame(fold_metrics)
display(fold_metrics_df)
fold_metrics_df.to_csv(OUTPUT_DIR / "fold_metrics.csv", index=False)

### OOF blend search

In [ ]:
oof["rank_ltr"] = within_state_rank(oof, "score_ltr")
oof["rank_bin"] = within_state_rank(oof, "score_bin")
oof["rank_cross"] = within_state_rank(oof, "cross_score")
oof["rank_graph"] = within_state_rank(oof, "score_graph")

blend_grid = []

for w_ltr in [0.45, 0.60, 0.75]:
    for w_bin in [0.00, 0.15, 0.25]:
        for w_cross in [0.00, 0.15, 0.30]:
            used = w_ltr + w_bin + w_cross
            if used > 0.95:
                continue
            w_graph = 1.0 - used

            score = (
                w_ltr * oof["rank_ltr"]
                + w_bin * oof["rank_bin"]
                + w_cross * oof["rank_cross"]
                + w_graph * oof["rank_graph"]
            )

            temp = oof.assign(score_blend=score)
            pred = top1_predictions(temp, "score_blend")
            acc = accuracy_from_predictions(pred, truth_map)

            blend_grid.append({
                "w_ltr": w_ltr,
                "w_bin": w_bin,
                "w_cross": w_cross,
                "w_graph": w_graph,
                "accuracy": acc,
            })

blend_results = pd.DataFrame(blend_grid).sort_values("accuracy", ascending=False)
BEST_BLEND = blend_results.iloc[0].to_dict()
display(blend_results.head(15))

oof["score_blend"] = (
    BEST_BLEND["w_ltr"] * oof["rank_ltr"]
    + BEST_BLEND["w_bin"] * oof["rank_bin"]
    + BEST_BLEND["w_cross"] * oof["rank_cross"]
    + BEST_BLEND["w_graph"] * oof["rank_graph"]
)

base_pred = top1_predictions(oof, "score_blend")
base_acc = accuracy_from_predictions(base_pred, truth_map)

print("Best OOF blend:", BEST_BLEND)
print("Base blend accuracy:", base_acc)

blend_results.to_csv(OUTPUT_DIR / "blend_search.csv", index=False)

### OOF override search

In [ ]:
def fold_pair_modal_lookup() -> dict[tuple[int, int], dict[str, float]]:
    lookup = {}
    for fold in range(CFG.N_FOLDS):
        fold_train = train_with_fold[train_with_fold.fold != fold]
        stats = (
            fold_train.groupby(["current_article_id", "target_article_id"])["next_article_id"]
            .agg(
                modal_next=lambda s: int(s.value_counts().index[0]),
                modal_count=lambda s: int(s.value_counts().iloc[0]),
                support="size",
            )
            .reset_index()
        )
        for row in stats.itertuples(index=False):
            lookup[(fold, int(row.current_article_id), int(row.target_article_id))] = {
                "candidate": int(row.modal_next),
                "share": float(row.modal_count / row.support),
                "support": int(row.support),
            }
    return lookup


pair_modal_oof = fold_pair_modal_lookup()

state_meta = (
    oof.groupby("state_id")
    .agg(
        fold=("fold", "first"),
        cur=("cur", "first"),
        tgt=("tgt", "first"),
    )
    .reset_index()
)

base_pred_map = dict(zip(base_pred.state_id.astype(int), base_pred.cand.astype(int)))
candidate_sets_oof = (
    oof.groupby("state_id")["cand"]
    .apply(lambda s: set(map(int, s)))
    .to_dict()
)

oof["confirmed_source"] = (
    (
        oof["source_ocr"]
        + oof["source_empirical"]
        + oof["source_sequence"]
    ) > 0
).astype(np.float32)

confirmed_oof = oof[oof["confirmed_source"] > 0].copy()
if len(confirmed_oof):
    confirmed_idx = confirmed_oof.groupby("state_id")["score_blend"].idxmax()
    best_confirmed_oof = dict(
        zip(
            confirmed_oof.loc[confirmed_idx, "state_id"].astype(int),
            confirmed_oof.loc[confirmed_idx, "cand"].astype(int),
        )
    )
else:
    best_confirmed_oof = {}

source_lookup_oof = {
    (int(sid), int(cand)): bool(confirmed)
    for sid, cand, confirmed in oof[
        ["state_id", "cand", "confirmed_source"]
    ].itertuples(index=False)
}

override_results = []

for pair_share_threshold in [0.60, 0.75, 0.90, 1.01]:
    for use_sequence in [False, True]:
        for use_direct_target in [False, True]:
            for use_confirmed_guard in [False, True]:
                pred_map = dict(base_pred_map)

                for row in state_meta.itertuples(index=False):
                    sid = int(row.state_id)
                    cur = int(row.cur)
                    tgt = int(row.tgt)
                    fold = int(row.fold)
                    candidates = candidate_sets_oof[sid]

                    pair_info = pair_modal_oof.get((fold, cur, tgt))
                    if (
                        pair_info is not None
                        and pair_info["support"] >= 2
                        and pair_info["share"] >= pair_share_threshold
                        and pair_info["candidate"] in candidates
                    ):
                        pred_map[sid] = pair_info["candidate"]

                    if use_sequence:
                        seq_candidate = SEQ_MAP.get(sid)
                        if seq_candidate in candidates:
                            pred_map[sid] = int(seq_candidate)

                    if use_direct_target and tgt in candidates:
                        pred_map[sid] = tgt

                    if (
                        use_confirmed_guard
                        and not source_lookup_oof.get((sid, int(pred_map[sid])), False)
                        and sid in best_confirmed_oof
                    ):
                        pred_map[sid] = int(best_confirmed_oof[sid])

                accuracy = np.mean([
                    pred_map[int(sid)] == int(actual)
                    for sid, actual in truth_map.items()
                ])

                override_results.append({
                    "pair_share_threshold": pair_share_threshold,
                    "use_sequence": use_sequence,
                    "use_direct_target": use_direct_target,
                    "use_confirmed_guard": use_confirmed_guard,
                    "accuracy": float(accuracy),
                })

override_results = pd.DataFrame(override_results).sort_values(
    "accuracy",
    ascending=False,
)

BEST_OVERRIDE = override_results.iloc[0].to_dict()
display(override_results.head(20))

print("Best override:", BEST_OVERRIDE)
print("Improvement over blend:", BEST_OVERRIDE["accuracy"] - base_acc)

override_results.to_csv(OUTPUT_DIR / "override_search.csv", index=False)

### Validation diagnostics

In [ ]:
best_pred_map = dict(base_pred_map)

for row in state_meta.itertuples(index=False):
    sid = int(row.state_id)
    cur = int(row.cur)
    tgt = int(row.tgt)
    fold = int(row.fold)
    candidates = candidate_sets_oof[sid]

    pair_info = pair_modal_oof.get((fold, cur, tgt))
    if (
        pair_info is not None
        and pair_info["support"] >= 2
        and pair_info["share"] >= BEST_OVERRIDE["pair_share_threshold"]
        and pair_info["candidate"] in candidates
    ):
        best_pred_map[sid] = pair_info["candidate"]

    if BEST_OVERRIDE["use_sequence"]:
        seq_candidate = SEQ_MAP.get(sid)
        if seq_candidate in candidates:
            best_pred_map[sid] = int(seq_candidate)

    if BEST_OVERRIDE["use_direct_target"] and tgt in candidates:
        best_pred_map[sid] = tgt

    if (
        BEST_OVERRIDE["use_confirmed_guard"]
        and not source_lookup_oof.get((sid, int(best_pred_map[sid])), False)
        and sid in best_confirmed_oof
    ):
        best_pred_map[sid] = int(best_confirmed_oof[sid])

diagnostics = train.copy()
diagnostics["pred"] = diagnostics.state_id.map(best_pred_map)
diagnostics["correct"] = (
    diagnostics.pred.astype(int) == diagnostics.next_article_id.astype(int)
).astype(int)
diagnostics["current_seen_multiple"] = diagnostics.current_article_id.map(
    train.current_article_id.value_counts()
).gt(1)
diagnostics["pair_support"] = diagnostics.set_index(
    ["current_article_id", "target_article_id"]
).index.map(
    train.groupby(["current_article_id", "target_article_id"]).size().to_dict()
)
diagnostics["sequence_available"] = diagnostics.state_id.isin(SEQ_MAP)
diagnostics["direct_target"] = (
    diagnostics.next_article_id == diagnostics.target_article_id
)

summary_rows = []
for segment, mask in {
    "all": np.ones(len(diagnostics), dtype=bool),
    "current_repeated": diagnostics.current_seen_multiple,
    "current_singleton": ~diagnostics.current_seen_multiple,
    "pair_repeated": diagnostics.pair_support > 1,
    "pair_singleton": diagnostics.pair_support <= 1,
    "sequence_available": diagnostics.sequence_available,
    "direct_target_truth": diagnostics.direct_target,
}.items():
    summary_rows.append({
        "segment": segment,
        "rows": int(mask.sum()),
        "accuracy": float(diagnostics.loc[mask, "correct"].mean()) if mask.sum() else np.nan,
    })

diagnostic_summary = pd.DataFrame(summary_rows)
display(diagnostic_summary)
diagnostic_summary.to_csv(OUTPUT_DIR / "oof_diagnostics.csv", index=False)

importance_df = pd.concat(feature_importance, ignore_index=True)
importance_summary = (
    importance_df.groupby("feature")["gain"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
display(importance_summary.head(25))
importance_summary.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)

## 10. Final training dan submission

In [ ]:
full_counts = fit_behavior_counts(train)
full_empirical = empirical_adjacency(train)

ptr_full = candidate_rows_for_states(
    train,
    full_empirical,
    with_label=True,
)
pte = candidate_rows_for_states(
    test,
    full_empirical,
    with_label=False,
)

ptr_full = complete_features(ptr_full, full_counts)
pte = complete_features(pte, full_counts)

final_ltr_rounds = max(
    100,
    int(np.median(best_ltr_iterations) * CFG.FINAL_ROUND_MULT),
)
final_bin_rounds = max(
    100,
    int(np.median(best_bin_iterations) * CFG.FINAL_ROUND_MULT),
)

print("Final LTR rounds:", final_ltr_rounds)
print("Final BIN rounds:", final_bin_rounds)

ptr_full = sort_pairs(ptr_full)
pte = sort_pairs(pte)

score_ltr = np.zeros(len(pte), dtype=np.float64)
score_bin = np.zeros(len(pte), dtype=np.float64)

for seed in CFG.FINAL_SEEDS:
    ltr_params = dict(CFG.LTR_PARAMS)
    ltr_params["seed"] = seed

    dtr_ltr = lgb.Dataset(
        ptr_full[FEATURES],
        label=ptr_full["label"],
        group=group_sizes(ptr_full),
        feature_name=FEATURES,
    )

    ltr_model = lgb.train(
        ltr_params,
        dtr_ltr,
        num_boost_round=final_ltr_rounds,
    )

    score_ltr += ltr_model.predict(pte[FEATURES]) / len(CFG.FINAL_SEEDS)

    bin_params = dict(CFG.BIN_PARAMS)
    bin_params["seed"] = seed + 1000

    dtr_bin = lgb.Dataset(
        ptr_full[FEATURES],
        label=ptr_full["label"],
        weight=inverse_group_weights(ptr_full),
        feature_name=FEATURES,
    )

    bin_model = lgb.train(
        bin_params,
        dtr_bin,
        num_boost_round=final_bin_rounds,
    )

    score_bin += bin_model.predict(pte[FEATURES]) / len(CFG.FINAL_SEEDS)

    del ltr_model, bin_model, dtr_ltr, dtr_bin
    gc.collect()

pte["score_ltr"] = score_ltr
pte["score_bin"] = score_bin
pte["score_graph"] = (
    -0.70 * pte["rank_dist_dir"]
    + 0.20 * pte["progress_dir"]
    + 0.10 * pte["rate_tc"]
    + 0.12 * pte["is_target"]
    + 0.15 * pte["seq_match"]
)

pte["rank_ltr"] = within_state_rank(pte, "score_ltr")
pte["rank_bin"] = within_state_rank(pte, "score_bin")
pte["rank_cross"] = within_state_rank(pte, "cross_score")
pte["rank_graph"] = within_state_rank(pte, "score_graph")

pte["score_final"] = (
    BEST_BLEND["w_ltr"] * pte["rank_ltr"]
    + BEST_BLEND["w_bin"] * pte["rank_bin"]
    + BEST_BLEND["w_cross"] * pte["rank_cross"]
    + BEST_BLEND["w_graph"] * pte["rank_graph"]
)

pred = top1_predictions(pte, "score_final")
pred_map = dict(zip(pred.state_id.astype(int), pred.cand.astype(int)))

pte["confirmed_source"] = (
    (
        pte["source_ocr"]
        + pte["source_empirical"]
        + pte["source_sequence"]
    ) > 0
).astype(np.float32)

confirmed_test = pte[pte["confirmed_source"] > 0].copy()
if len(confirmed_test):
    confirmed_idx_test = confirmed_test.groupby("state_id")["score_final"].idxmax()
    best_confirmed_test = dict(
        zip(
            confirmed_test.loc[confirmed_idx_test, "state_id"].astype(int),
            confirmed_test.loc[confirmed_idx_test, "cand"].astype(int),
        )
    )
else:
    best_confirmed_test = {}

source_lookup_test = {
    (int(sid), int(cand)): bool(confirmed)
    for sid, cand, confirmed in pte[
        ["state_id", "cand", "confirmed_source"]
    ].itertuples(index=False)
}

candidate_sets_test = (
    pte.groupby("state_id")["cand"]
    .apply(lambda s: set(map(int, s)))
    .to_dict()
)

full_pair_stats = (
    train.groupby(["current_article_id", "target_article_id"])["next_article_id"]
    .agg(
        modal_next=lambda s: int(s.value_counts().index[0]),
        modal_count=lambda s: int(s.value_counts().iloc[0]),
        support="size",
    )
    .reset_index()
)
full_pair_lookup = {
    (int(r.current_article_id), int(r.target_article_id)): {
        "candidate": int(r.modal_next),
        "share": float(r.modal_count / r.support),
        "support": int(r.support),
    }
    for r in full_pair_stats.itertuples(index=False)
}

for row in test.itertuples(index=False):
    sid = int(row.state_id)
    cur = int(row.current_article_id)
    tgt = int(row.target_article_id)
    candidates = candidate_sets_test[sid]

    pair_info = full_pair_lookup.get((cur, tgt))
    if (
        pair_info is not None
        and pair_info["support"] >= 2
        and pair_info["share"] >= BEST_OVERRIDE["pair_share_threshold"]
        and pair_info["candidate"] in candidates
    ):
        pred_map[sid] = pair_info["candidate"]

    if BEST_OVERRIDE["use_sequence"]:
        seq_candidate = SEQ_MAP.get(sid)
        if seq_candidate in candidates:
            pred_map[sid] = int(seq_candidate)

    if BEST_OVERRIDE["use_direct_target"] and tgt in candidates:
        pred_map[sid] = tgt

    if (
        BEST_OVERRIDE["use_confirmed_guard"]
        and not source_lookup_test.get((sid, int(pred_map[sid])), False)
        and sid in best_confirmed_test
    ):
        pred_map[sid] = int(best_confirmed_test[sid])

submission = sample_submission[["state_id"]].copy()
submission["predicted_next_article_id"] = (
    submission.state_id.map(pred_map).astype(int)
)

assert len(submission) == len(sample_submission)
assert submission.predicted_next_article_id.notna().all()
assert submission.predicted_next_article_id.isin(article_ids).all()

submission_path = OUTPUT_DIR / "submission_wikinav_v2.csv"
submission.to_csv(submission_path, index=False)

display(submission.head())
print("Saved:", submission_path)

### Submission source audit

In [ ]:
state_lookup_test = test.set_index("state_id")

audit_rows = []
for row in submission.itertuples(index=False):
    sid = int(row.state_id)
    pred_id = int(row.predicted_next_article_id)
    current = int(state_lookup_test.loc[sid, "current_article_id"])

    audit_rows.append({
        "state_id": sid,
        "prediction": pred_id,
        "confirmed_ocr": float(pred_id in OCR_ADJ.get(current, set())),
        "confirmed_empirical": float(pred_id in full_empirical.get(current, set())),
        "sequence_candidate": float(SEQ_MAP.get(sid) == pred_id),
    })

submission_audit = pd.DataFrame(audit_rows)
display(submission_audit.mean(numeric_only=True).to_frame("mean"))
submission_audit.to_csv(OUTPUT_DIR / "submission_source_audit.csv", index=False)

## 11. Error analysis dan output bundle

In [ ]:
report_lines = [
    "# WikiNav V2 Run Summary",
    "",
    f"- Data directory: `{DATA_DIR}`",
    f"- OCR pages: {len(OCR_RECORDS)}",
    f"- OCR edges: {sum(len(v) for v in OCR_ADJ.values())}",
    f"- OCR-only train label recall: {ocr_train_recall:.6f}",
    f"- Selected sequence offset: {BEST_SEQ_OFFSET}",
    f"- Sequence precision: {SEQ_RULE_PRECISION:.6f}",
    f"- Mean fold candidate recall: {fold_metrics_df.candidate_recall.mean():.6f}",
    f"- Mean fold LTR accuracy: {fold_metrics_df.ltr_accuracy.mean():.6f}",
    f"- Mean fold binary accuracy: {fold_metrics_df.binary_accuracy.mean():.6f}",
    f"- Best blend OOF accuracy: {base_acc:.6f}",
    f"- Best routed OOF accuracy: {BEST_OVERRIDE['accuracy']:.6f}",
    "",
    "## Best blend",
    "",
    "```json",
    json.dumps(BEST_BLEND, indent=2),
    "```",
    "",
    "## Best override",
    "",
    "```json",
    json.dumps(BEST_OVERRIDE, indent=2),
    "```",
]

run_report_path = OUTPUT_DIR / "RUN_SUMMARY.md"
run_report_path.write_text("\n".join(report_lines), encoding="utf-8")

bundle_path = Path("/kaggle/working/wikinav_v2_bundle.zip")
if not Path("/kaggle/working").exists():
    bundle_path = OUTPUT_DIR.parent / "wikinav_v2_bundle.zip"

import zipfile

with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=f"wikinav_v2/{path.relative_to(OUTPUT_DIR)}")

print("Run report:", run_report_path)
print("Bundle    :", bundle_path)
print("Submission:", submission_path)

## Interpretation order

Setelah Run All selesai, baca hasil dalam urutan:

1. `state_sequence_audit.csv`
2. `ocr_summary.csv`
3. `fold_metrics.csv`
4. `blend_search.csv`
5. `override_search.csv`
6. `oof_diagnostics.csv`
7. `submission_source_audit.csv`
8. `RUN_SUMMARY.md`

Jangan submit hanya karena notebook selesai. Pastikan candidate recall, OOF accuracy, dan source audit masuk akal.